In [6]:
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

In [2]:
tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")
model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER")
nlp = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3503.98it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/nmi/projects/grassroot_drones/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:188: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torc

In [8]:
df = pd.read_csv("../data/processed/scraped_with_topics_snippets_only.csv")

def extract_entities(text):
    if not isinstance(text, str):
        return "[]"

    results = nlp(text[:1500]) 
    
    clean_results = [
        {
            "entity": ent['entity_group'], 
            "word": ent['word'], 
            "score": round(float(ent['score']), 4)
        } 
        for ent in results
    ]
    
    return json.dumps(clean_results)

print("Extracting entities...")
df['raw_entities'] = df['snippet'].apply(extract_entities)

output_path = "../data/processed/scraped_with_raw_entities.csv"
df.to_csv(output_path, index=False)
print(f"Saved raw results to {output_path}")

print("\nEntities found in the first report:")
print(df['raw_entities'].iloc[0])



Extracting entities...
Saved raw results to ../data/processed/scraped_with_raw_entities.csv

Entities found in the first report:
[{"entity": "LOC", "word": "US", "score": 0.9443}, {"entity": "PER", "word": "Marco Rubio", "score": 0.9995}, {"entity": "MISC", "word": "Russian", "score": 0.9998}, {"entity": "LOC", "word": "United States", "score": 0.9995}, {"entity": "LOC", "word": "Russia", "score": 0.9997}, {"entity": "LOC", "word": "Russia", "score": 0.9998}, {"entity": "LOC", "word": "Ukraine", "score": 0.9998}, {"entity": "MISC", "word": "Alaska Summit", "score": 0.9453}, {"entity": "MISC", "word": "Ukrainian", "score": 0.9997}, {"entity": "PER", "word": "Vol", "score": 0.9997}, {"entity": "PER", "word": "##odymyr Zelensky", "score": 0.9299}, {"entity": "LOC", "word": "United States", "score": 0.9996}, {"entity": "LOC", "word": "Russia", "score": 0.9997}, {"entity": "MISC", "word": "Ukrainian", "score": 0.9997}, {"entity": "MISC", "word": "Russian", "score": 0.9998}, {"entity": "MISC

In [9]:
def clean_and_format_entities(json_str):
    if pd.isna(json_str):
        return ""
    
    try:
        entities = json.loads(json_str)
    except json.JSONDecodeError:
        return ""
        
    category_map = {
        'PER': 'People',
        'LOC': 'Locations',
        'ORG': 'Organizations',
        'MISC': 'Miscellaneous'
    }
    
    cleaned = []
    
    for item in entities:
        word = item['word']
        ent_type = item['entity'].replace('B-', '').replace('I-', '') 
        
        if word.startswith('##') and cleaned and cleaned[-1]['entity'] == ent_type:
            cleaned[-1]['word'] += word.replace('##', '')
        else:
            cleaned.append({'entity': ent_type, 'word': word})
            
    grouped = {}
    for item in cleaned:
        cat = category_map.get(item['entity'], 'Other')
        word = item['word'].strip()
        
        if cat not in grouped:
            grouped[cat] = []
        if word not in grouped[cat]:
            grouped[cat].append(word)
            
    formatted_parts = []
    for cat, words in grouped.items():
        formatted_parts.append(f"{cat}: {', '.join(words)}")
        
    return " | ".join(formatted_parts)

print("Formatting entities...")
df['formatted_entities'] = df['raw_entities'].apply(clean_and_format_entities)

print("\nFormatted Output for Report 1:")
print(df['formatted_entities'].iloc[0])

output_path = "../data/processed/scraped_with_final_entities.csv"
df.to_csv(output_path, index=False)
print(f"\nSaved final results to {output_path}")

Formatting entities...

Formatted Output for Report 1:
Locations: US, United States, Russia, Ukraine | People: Marco Rubio, Volodymyr Zelensky | Miscellaneous: Russian, Alaska Summit, Ukrainian, Iskander, M, Kh

Saved final results to ../data/processed/scraped_with_final_entities.csv


In [ ]:
df.to_csv("../data/processed/scraped_with_entities.csv", index=False)